# HS4002 Week 11

## Panel Data Models

Panel data follows the **same units** (people, players, countries) across multiple time points, allowing us to control for unobserved unit-level heterogeneity.

In Python, the `linearmodels` library provides panel regression — equivalent to R's `plm` package.

We'll use NBA player data (2013–2017) to ask: **how does a player's age affect their scoring performance?**

In [3]:
%pip install linearmodels


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
# !pip install pandas numpy plotnine linearmodels stargazer

import pandas as pd
import numpy as np
from plotnine import *
import warnings
warnings.filterwarnings('ignore')
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS

## Setup: Load and Prepare the NBA Dataset

In [5]:
# Data source: https://www.kaggle.com/datasets/drgilermo/nba-players-stats
# NBA player statistics 1950-2017 from basketball-reference.com
raw_df = pd.read_csv("bball.csv").iloc[:, 1:]  # drop the unnamed index column

df = (
	raw_df[raw_df["Year"] >= 2013]
	.groupby(["Player", "Year"])
	.agg(
		games_played    = ("G",    "sum"),
		games_started   = ("GS",   "sum"),
		minutes_played  = ("MP",   "sum"),
		age             = ("Age",  "mean"),
		personal_fouls  = ("PF",   "sum"),
		points          = ("PTS",  "sum"),
		true_shooting   = ("TS%",  "mean"),
		assists         = ("AST",  "sum"),
		steals          = ("STL",  "sum"),
		blocks          = ("BLK",  "sum"),
		fga             = ("FGA",  "sum"),
		fg              = ("FG",   "sum"),
		threeP          = ("3P",   "sum"),
		threePA         = ("3PA",  "sum"),
		twoP            = ("2P",   "sum"),
		twoPA           = ("2PA",  "sum"),
		ft              = ("FT",   "sum"),
		fta             = ("FTA",  "sum"),
		def_rb          = ("DRB",  "sum"),
		off_rb          = ("ORB",  "sum"),
	)
	.assign(ppg=lambda x: x["points"] / x["games_played"])
	.reset_index()
)
print(df.shape)
df.head()

(2403, 23)


,Player,Year,games_played,games_started,minutes_played,age,personal_fouls,points,true_shooting,assists,...,fg,threeP,threePA,twoP,twoPA,ft,fta,def_rb,off_rb,ppg
0,A.J. Hammons,2017.0,22.0,0.0,163.0,24.0,21.0,48.0,0.472000,4.0,...,17.0,5.0,10.0,12.0,32.0,9.0,20.0,28.0,8.0,2.181818
1,A.J. Price,2013.0,57.0,22.0,1278.0,26.0,73.0,441.0,0.501000,205.0,...,161.0,70.0,200.0,91.0,213.0,49.0,62.0,94.0,20.0,7.736842
2,A.J. Price,2014.0,28.0,0.0,99.0,27.0,5.0,44.0,0.469000,13.0,...,19.0,6.0,22.0,13.0,24.0,0.0,2.0,9.0,1.0,1.571429
3,A.J. Price,2015.0,52.0,0.0,648.0,28.0,30.0,266.0,0.376750,92.0,...,102.0,30.0,114.0,72.0,160.0,32.0,48.0,52.0,12.0,5.115385
4,Aaron Brooks,2013.0,106.0,40.0,1994.0,28.0,188.0,752.0,0.500667,228.0,...,286.0,100.0,268.0,186.0,364.0,80.0,104.0,134.0,26.0,7.094340


## Setting up the Panel Index

`linearmodels` requires a **MultiIndex** of (entity, time). Equivalent to the `index=c('Player','Year')` argument in R's `plm()`.

In [6]:
# Set the panel index — required by linearmodels
df_panel = df.set_index(['Player', 'Year'])

# A Tutorial for Panel Models

## Pooled OLS

Equivalent to a regular `lm()` in R — ignores the panel structure.

In [7]:
pool_model = PooledOLS.from_formula('ppg ~ age + I(age**2)', data=df_panel)
pool_result = pool_model.fit()
print(pool_result.summary)

                          PooledOLS Estimation Summary                          
Dep. Variable:                    ppg   R-squared:                        0.6757
Estimator:                  PooledOLS   R-squared (Between):              0.6103
No. Observations:                2403   R-squared (Within):               0.0318
Date:                Thu, Nov 05 2026   R-squared (Overall):              0.6757
Time:                        09:59:00   Log-likelihood                   -7587.5
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      2501.2
Entities:                         817   P-value                           0.0000
Avg Obs:                       2.9412   Distribution:                  F(2,2401)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             2501.2
                            

## Fixed Effects (Within) Model

Adds `EntityEffects` to control for time-invariant player characteristics. Equivalent to `plm(..., model='within')` in R.

In [8]:
fe_model = PanelOLS.from_formula('ppg ~ age + I(age**2) + EntityEffects', data=df_panel)
fe_result = fe_model.fit()
print(fe_result.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                    ppg   R-squared:                        0.1774
Estimator:                   PanelOLS   R-squared (Between):             -66.812
No. Observations:                2403   R-squared (Within):               0.1774
Date:                Thu, Nov 05 2026   R-squared (Overall):             -43.473
Time:                        09:59:00   Log-likelihood                   -5167.4
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      170.80
Entities:                         817   P-value                           0.0000
Avg Obs:                       2.9412   Distribution:                  F(2,1584)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             170.80
                            

Interpret the coefficients. How does the fixed-effects estimate differ from the pooled OLS estimate?

## Random Effects Model

Equivalent to `plm(..., model='random')` in R.

In [9]:
re_model = RandomEffects.from_formula('ppg ~ age + I(age**2)', data=df_panel)
re_result = re_model.fit()
print(re_result.summary)

                        RandomEffects Estimation Summary                        
Dep. Variable:                    ppg   R-squared:                        0.3804
Estimator:              RandomEffects   R-squared (Between):              0.6371
No. Observations:                2403   R-squared (Within):               0.0412
Date:                Thu, Nov 05 2026   R-squared (Overall):              0.6573
Time:                        09:59:00   Log-likelihood                   -5810.5
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      737.02
Entities:                         817   P-value                           0.0000
Avg Obs:                       2.9412   Distribution:                  F(2,2401)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             737.02
                            

## Model Diagnostics

### Lagrange Multiplier Test

Tests whether individual effects are present (i.e., whether FE/RE is needed over pooled OLS). Equivalent to R's `plmtest(fe_model, effect='individual')`.

In [10]:
# Breusch-Pagan LM test for random effects
# Under H0 (no individual effects), we prefer pooled OLS.
# Equivalent to R's plmtest(..., effect='individual', type='bp').
from scipy.stats import chi2

resid = pool_result.resids
e_i   = resid.groupby(level=0).sum()
ssr   = float((resid ** 2).sum())
A1    = float((e_i ** 2).sum()) / ssr - 1
T_i   = resid.groupby(level=0).size()
n_obs = resid.shape[0]
lm_stat = (n_obs ** 2) / (2 * (float((T_i ** 2).sum()) - n_obs)) * A1 ** 2
lm_pval = chi2.sf(lm_stat, df=1)

print('LM test for individual effects:')
print(f'LM statistic = {lm_stat:.4f}')
print('df = 1')
print(f'p-value = {lm_pval:.4f}')

LM test for individual effects:
LM statistic = 2381.2182
df = 1
p-value = 0.0000


### Hausman Test

Tests whether RE estimates are consistent (i.e., whether FE or RE is preferred). Under H0, RE is consistent and more efficient. Equivalent to R's `phtest(fe_model, re_model)`.

In [11]:
# Hausman test: compare FE and RE estimates
# Under H0 (RE consistent), the difference in coefficients should be small
from scipy.stats import chi2

common_params = fe_result.params.index.intersection(re_result.params.index)

beta_diff = fe_result.params[common_params] - re_result.params[common_params]
cov_diff  = (
	fe_result.cov.loc[common_params, common_params] -
	re_result.cov.loc[common_params, common_params]
)

h_stat = float(beta_diff.T @ np.linalg.inv(cov_diff.values) @ beta_diff)
p_val  = chi2.sf(h_stat, df=len(common_params))
print(f'Hausman statistic = {h_stat:.4f}')
print(f'df = {len(common_params)}')
print(f'p-value = {p_val:.4f}')
print('Prefer FE over RE:', p_val < 0.05)

Hausman statistic = 285.0094
df = 2
p-value = 0.0000
Prefer FE over RE: True


## Comparing the Models

As in earlier weeks, `stargazer` collects the fitted models into one regression table. Reading pooled OLS, fixed effects, and random effects side by side is easier than scrolling through three summaries. Equivalent to R's `stargazer(lm1, fe_glm1, re_glm1)`.

In [12]:
from stargazer.stargazer import Stargazer

sg = Stargazer([pool_result, fe_result, re_result])
sg.title('Points per Game and Age')

# Label the models and variables readably, rather than by raw column name
sg.dependent_variable_name('Points per game')
sg.custom_columns(['Pooled OLS', 'Fixed Effects', 'Random Effects'], [1, 1, 1])
sg.covariate_order(['age', 'I(age ** 2)'])
sg.rename_covariates({
	'age':         'Age',
	'I(age ** 2)': 'Age-squared',
})

# Ending the cell on the object itself renders the table in the notebook
sg

# Independent Analysis: Has Sports Science Extended Career Peaks?

Compare players from 2008–2012 with players from 2013–2017. Does age affect scoring differently across the two eras?

In [13]:
# (1) create two data sets, one for each era.

def make_panel(start, end):
    d = (
        raw_df[raw_df["Year"].between(start, end)]
        .groupby(["Player", "Year"])
        .agg(
            games_played = ("G",   "sum"),
            points       = ("PTS", "sum"),
            age          = ("Age", "mean"),
        )
        .assign(ppg=lambda x: x["points"] / x["games_played"])
        .reset_index()
    )
    return d.set_index(["Player", "Year"])

panel_0812 = make_panel(2008, 2012)
panel_1317 = make_panel(2013, 2017)
print(panel_0812.shape, panel_1317.shape)

(2266, 4) (2403, 4)


In [ ]:
# (2) fit models for each of the two data sets

def fit_models(panel):
    pool = PooledOLS.from_formula('ppg ~ age + I(age**2)', data=panel).fit()
    fe   = PanelOLS.from_formula('ppg ~ age + I(age**2) + EntityEffects', data=panel).fit()
    re   = RandomEffects.from_formula('ppg ~ age + I(age**2)', data=panel).fit()
    return pool, fe, re

pool_0812, fe_0812, re_0812 = fit_models(panel_0812)
pool_1317, fe_1317, re_1317 = fit_models(panel_1317)

def lm_test(pool_res):
    resid = pool_res.resids
    e_i = resid.groupby(level=0).sum()
    ssr = float((resid ** 2).sum())
    A1 = float((e_i ** 2).sum()) / ssr - 1
    T_i = resid.groupby(level=0).size()
    n = resid.shape[0]
    stat = (n ** 2) / (2 * (float((T_i ** 2).sum()) - n)) * A1 ** 2
    print(f'LM test: stat = {stat:.2f}, p = {chi2.sf(stat, df=1):.4f}')

def hausman(fe_res, re_res):
    p = fe_res.params.index.intersection(re_res.params.index)
    b = fe_res.params[p] - re_res.params[p]
    V = fe_res.cov.loc[p, p] - re_res.cov.loc[p, p]
    stat = float(b.T @ np.linalg.inv(V.values) @ b)
    pval = chi2.sf(stat, df=len(p))
    print(f'Hausman: stat = {stat:.2f}, p = {pval:.4f}, prefer FE: {pval < 0.05}')

def peak_age(res):
    return -res.params['age'] / (2 * res.params['I(age ** 2)'])

In [19]:
# (3) run diagnostics, compare, and interpret for 2008-2012.

lm_test(pool_0812)
hausman(fe_0812, re_0812)
print('Peak age (FE):', round(peak_age(fe_0812), 2))

sg = Stargazer([pool_0812, fe_0812, re_0812])
sg.title('Points per Game and Age, 2008-2012')
sg.custom_columns(['Pooled OLS', 'Fixed Effects', 'Random Effects'], [1, 1, 1])
sg

LM test: stat = 2458.00, p = 0.0000
Hausman: stat = 271.53, p = 0.0000, prefer FE: True
Peak age (FE): 24.95


Interpretation: The LM test is highly significant (LM = 2458.00, p < 0.001), so there are significant player-level effects and pooled OLS is not appropriate. The Hausman test is also significant (χ² = 271.53, p < 0.001), which means the random-effects estimates are inconsistent, so fixed effects is the preferred model.

In the FE model, age has a positive linear coefficient (5.094) and a negative squared term (−0.102), both significant at p < 0.01. This gives an inverted-U: as a given player gets older, his scoring first rises and then falls. Scoring peaks at about age 25 (5.094 ÷ (2 × 0.102) ≈ 24.95). Around age 22, a player gains roughly 0.6 ppg per year; by age 28, he loses roughly 0.6 ppg per year.

The pooled OLS and RE coefficients are much smaller (age = 0.687 and 0.835) because they mix comparisons between different players with changes within the same player. Players who last into their 30s tend to be the better scorers, which flattens the curve. FE compares each player only with himself, so it isolates the effect of ageing.

In [20]:
# (4) run diagnostics, compare, and interpret for 2013-2017.

lm_test(pool_1317)
hausman(fe_1317, re_1317)
print('Peak age (FE):', round(peak_age(fe_1317), 2))

sg = Stargazer([pool_1317, fe_1317, re_1317])
sg.title('Points per Game and Age, 2013-2017')
sg.custom_columns(['Pooled OLS', 'Fixed Effects', 'Random Effects'], [1, 1, 1])
sg

LM test: stat = 2381.22, p = 0.0000
Hausman: stat = 285.01, p = 0.0000, prefer FE: True
Peak age (FE): 26.64


Interpretation: Again, the LM test is significant (LM = 2381.22, p < 0.001), so player effects are present, and the Hausman test is significant (χ² = 285.01, p < 0.001), so fixed effects is preferred over random effects.

The FE model again shows an inverted-U: age = 5.745 and age² = −0.108, both p < 0.01. Scoring peaks at about age 26.6. Around age 22, a player gains roughly 1 ppg per year; by age 30, he loses roughly 0.7 ppg per year. As in the earlier era, pooled OLS and RE give much flatter curves than FE, because they are influenced by differences between players rather than changes within each player.

# (5) what differences do you find in the age-scoring relationship across the two eras?
Both the LM and Hausman tests are significant in both eras, so fixed effects is preferred.

In both eras, the FE models show an inverted-U relationship: as a player ages, his points per game rise, peak, and then decline. However, the peak comes later in 2013–2017 (about age 26.6) than in 2008–2012 (about age 25.0), a shift of roughly 1.7 years. The age-squared coefficients are very similar (−0.102 vs −0.108), so players do not decline more slowly after their peak. Instead, the whole curve has shifted to later ages.

This is consistent with sports science helping players stay at their best for longer, but we cannot attribute the shift to sports science alone. League-wide scoring rose over 2013–2017, and because age and year increase together, the model may partly read this trend as an age effect. The move towards three-point shooting may also favour skills that age better than athleticism, and changes in which players stay in the league, or how their minutes are managed, could shift the curve too. Finally, we compared two separate models rather than formally testing whether the peak ages differ, so the result suggests, rather than proves, that career peaks have been extended.